# Inputs: prompts, tools, and data

A JAZ agent is launched through `invoke`. You call `invoke` with some inputs as
keyword arguments, and the agent writes code in a Python REPL loop until it returns.
JAZ makes no distinction between different kinds of inputs. Your prompt, tools,
data, and any other Python object you give the agent are all treated the same way:
they become variables in the REPL and are shown in the agent's prompt.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [ ]:
%pip install -q jaz-lang pandas

In [1]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [2]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## All inputs are born equal

Inputs to `invoke` are **arbitrary keyword arguments**: `invoke(name1=value1, name2=value2, ...)`.
An input can be any Python object and you can give it whatever name you want.

Every keyword argument you pass to `invoke`:

1. becomes a **variable in the agent's REPL**, under the name you gave it, and
2. is **shown in the agent's prompt**, so the model sees it.

*JAZ has no notion of a “prompt” argument, a “tool” argument, or a “data” argument.* There is no such distinction to make, because *no distinction is needed*.

That is the whole input model. No schemas, no registration, no JSON.

To understand this, we need a helper hook that lets us inspect the prompt and agent trajectory.
*You do not need to understand it now - hooks are introduced in [tutorial 3](03-hooks-events-and-effects.ipynb)*.

In [3]:
from jaz import invoke
from jaz.hooks import Hook


class Logger(Hook):
    """Prints the prompt the agent is given, so we can see it.

    You do NOT need to understand this class to follow this tutorial. It is scaffolding:
    it lets us look at what the model was handed. Hooks are the subject of tutorial 3.
    """

    def on_llm_query_enter(self, event):
        user_message = event.messages[-1]
        assert user_message["role"] == "user"
        label = "USER PROMPT" if event.iteration == 0 else "REPL OUTPUT"
        print(f"--- {label} BEGINS ---")
        print(user_message["content"])
        print(f"--- {label} ENDS ---")

    def on_llm_query_complete(self, event):
        print("--- LLM RESPONSE BEGINS ---")
        print(event.response.content)
        print("--- LLM RESPONSE ENDS ---")

Now a call with two inputs: a string, and a pandas DataFrame of 344 penguins of three species,
measured on three islands of the Palmer Archipelago in Antarctica (seaborn's copy of the
[Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) data). Every tutorial in this
series works on this same table, and it is the one on the [JAZ homepage](https://jaz-lang.ai).

In [4]:
import pandas as pd

# Palmer Penguins: 344 penguins, one row each.
PENGUINS = (
    "https://raw.githubusercontent.com"
    "/mwaskom/seaborn-data/master/penguins.csv"
)
penguins = pd.read_csv(PENGUINS)

with Logger():
    answer = invoke(
        task="Compute the average body mass of each species.",
        penguins=penguins,
    )

answer

--- USER PROMPT BEGINS ---
<task type="str">
Compute the average body mass of each species.
</task>
<penguins type="DataFrame">
    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
3    Adelie  Torgersen             NaN            NaN                NaN   
4    Adelie  Torgersen            36.7           19.3              193.0   
..      ...        ...             ...            ...                ...   
339  Gentoo     Biscoe             NaN            NaN                NaN   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8              212.0   
343  Gentoo     Biscoe            49

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute average body_mass_g per species (ignoring missing values) and return as a mapping.
result = (
    penguins.groupby("species")["body_mass_g"]
    .mean(numeric_only=True)
    .to_dict()
)
return result
--- LLM RESPONSE ENDS ---


{'Adelie': 3700.662251655629,
 'Chinstrap': 3733.0882352941176,
 'Gentoo': 5076.016260162602}

Read the prompt above. `task` and `penguins` are listed the *same way*, each with its name,
type, and content. Nothing in the framework marks one as the instruction and the other as the
data — the agent simply received two named values and worked out what to do. It saw a section
in its prompt titled `<task>` and interpreted its contents as the task, and it saw a section
named `<penguins>` and interpreted it as the data the task was referring to.

The DataFrame is shown the way pandas prints it, abridged to its first and last rows. The agent
does not need the rest in its prompt: `penguins` is the real DataFrame in its REPL, so it
called pandas on all 344 rows.

`task` is not a keyword JAZ knows. It is a string that happens to be called `task` — you can
use any other name that expresses the same meaning (e.g., `question` or `instructions`).

In [5]:
# Giving different names yields the same result - it just renames the section titles
# in the prompt and the variables in the REPL.
with Logger():
    answer = invoke(
        question="What is the mean body mass of each species?",
        data=penguins,
    )

answer

--- USER PROMPT BEGINS ---
<question type="str">
What is the mean body mass of each species?
</question>
<data type="DataFrame">
    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
3    Adelie  Torgersen             NaN            NaN                NaN   
4    Adelie  Torgersen            36.7           19.3              193.0   
..      ...        ...             ...            ...                ...   
339  Gentoo     Biscoe             NaN            NaN                NaN   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8              212.0   
343  Gentoo     Biscoe            4

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute the mean body mass (body_mass_g) for each species, ignoring missing values.
import pandas as pd

result = (
    data.groupby("species", dropna=False)["body_mass_g"]
    .mean()
    .sort_index()
)

return result
--- LLM RESPONSE ENDS ---
--- REPL OUTPUT BEGINS ---
ImportError: Import of module 'pandas' is forbidden in the REPL.

--- REPL OUTPUT ENDS ---


--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute mean body mass by species without importing pandas.
# We'll iterate over rows in the provided DataFrame-like object and aggregate sums/counts.

rows = data.to_dict(orient="records")

sums = {}
counts = {}

for r in rows:
    sp = r.get("species")
    bm = r.get("body_mass_g")
    if sp is None:
        continue
    if bm is None or (isinstance(bm, float) and bm != bm):  # NaN check
        continue
    sums[sp] = sums.get(sp, 0.0) + float(bm)
    counts[sp] = counts.get(sp, 0) + 1

means = {sp: (sums[sp] / counts[sp] if counts[sp] else float("nan")) for sp in sums}

# Sort by species name for stable output
result = {k: means[k] for k in sorted(means.keys())}

return result
--- LLM RESPONSE ENDS ---


{'Adelie': 3700.662251655629,
 'Chinstrap': 3733.0882352941176,
 'Gentoo': 5076.016260162602}

## A tool is just an input you can call

Nothing new is needed for tools. Pass the function.

The means are one line of pandas, so the agent wrote them itself. Whether two species really
differ is a harder question: Adelie and Chinstrap penguins are only about 30 g apart on average,
and telling a real difference from chance takes a statistical test. The agent's REPL cannot
import anything — not even `math` — so it would have to write that test from scratch. Here it
comes as a tool instead.

Notice the prompt never mentions the function, because the very act
of passing in the `same_distribution_p_value` function renders it in the user prompt.

In [6]:
import math

def same_distribution_p_value(a: list[float], b: list[float]) -> float:
    """Mann-Whitney U test: the p-value that two samples come from the same
    distribution. Missing values (NaN) are ignored."""
    a, b = [x for x in a if x == x], [y for y in b if y == y]
    n1, n2 = len(a), len(b)
    u = sum((x > y) + (x == y) / 2 for x in a for y in b)
    z = (abs(u - n1 * n2 / 2) - 0.5) / math.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    return math.erfc(z / math.sqrt(2))


with Logger():
    reply = invoke(
        prompt="How likely is it that Adelie and Chinstrap penguins have "
               "the same body mass distribution?",
        penguins=penguins,
        same_distribution_p_value=same_distribution_p_value,
    )

reply

--- USER PROMPT BEGINS ---
<prompt type="str">
How likely is it that Adelie and Chinstrap penguins have the same body mass distribution?
</prompt>
<penguins type="DataFrame">
    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
3    Adelie  Torgersen             NaN            NaN                NaN   
4    Adelie  Torgersen            36.7           19.3              193.0   
..      ...        ...             ...            ...                ...   
339  Gentoo     Biscoe             NaN            NaN                NaN   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8            

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute Mann-Whitney U p-value comparing body_mass_g distributions for Adelie vs Chinstrap; ignore NaNs.
import math

adelie = []
chinstrap = []

for _, row in penguins.iterrows():
    sp = row["species"]
    bm = row["body_mass_g"]
    # Ignore missing values
    if bm is None:
        continue
    try:
        if math.isnan(bm):
            continue
    except TypeError:
        # If bm isn't a float, skip
        continue
    if sp == "Adelie":
        adelie.append(float(bm))
    elif sp == "Chinstrap":
        chinstrap.append(float(bm))

if len(adelie) == 0 or len(chinstrap) == 0:
    raise Exception("Insufficient data: Adelie or Chinstrap has no non-missing body_mass_g values.")

p_value = same_distribution_p_value(adelie, chinstrap)

print("N Adelie (non-missing):", len(adelie))
print("N Chinstrap (non-missing):", len(chinstrap))
print("Mann-Whitney U p-value (same distribution):", p_value)

return p_value
--- LLM RESPO

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Recompute p-value without importing math; filter out NaNs using float('nan') behavior and/or NaN != NaN.
adelie = []
chinstrap = []

for _, row in penguins.iterrows():
    sp = row["species"]
    bm = row["body_mass_g"]

    if bm is None:
        continue

    # NaN check without importing anything: NaN is the only float value that is not equal to itself
    try:
        if float(bm) != float(bm):
            continue
    except Exception:
        continue

    bm = float(bm)

    if sp == "Adelie":
        adelie.append(bm)
    elif sp == "Chinstrap":
        chinstrap.append(bm)

if len(adelie) == 0 or len(chinstrap) == 0:
    raise Exception("Insufficient data: Adelie or Chinstrap has no non-missing body_mass_g values.")

p_value = same_distribution_p_value(adelie, chinstrap)

print("N Adelie (non-missing):", len(adelie))
print("N Chinstrap (non-missing):", len(chinstrap))
print("Mann-Whitney U p-value (same distribution):"

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Return the already-computed p-value without printing (per REPL rule).
p_value = 0.4856732660841894
return p_value
--- LLM RESPONSE ENDS ---


0.4856732660841894

Look at how `same_distribution_p_value` was shown to the model: its **signature and docstring**.
That is why a tool needs no schema — a Python function already carries its own description,
and the agent decided by itself to call it.

(The answer, about 0.49, says the two species' weights are entirely consistent with being drawn
from the same distribution: on body mass alone, you cannot tell an Adelie from a Chinstrap.)

## How an input is displayed to the model

What the model sees depends on what kind of object the input is:

| The input is… | The model is shown |
| :------------ | :----------------- |
| a function or method | its signature and docstring |
| a class | its docstring, then its public members |
| an object that does not print itself | its class docstring, then its public members |
| a module | its summary and the names it exports |
| anything else (the default) | its `str()` — the value itself |

**Public members** means public methods with their signature and docstring, and public
attributes with their name and type — the same information you would get from `help()`.

The third row has a condition worth pausing on: *does not print itself*. If an object already
knows how to print itself — a list, a number, a `DataFrame` — then printing it is what the
model wants. The table above told the agent far more than a list of the hundreds of methods on
`DataFrame` would. JAZ falls back to listing the public members only when the object has no
printed form of its own, which is the usual case for a class you wrote yourself.

You have already seen two of these rows: `penguins` printed as its table, and
`same_distribution_p_value` as a signature. Here is the third — an ordinary object that carries
some data *and* a way to use it:

In [7]:
class Colony:
    """The penguins observed on one island."""

    def __init__(self, island: str, penguins: pd.DataFrame):
        self.island = island
        self.penguins = penguins

    def share(self, species: str) -> float:
        """Percentage of this colony's penguins that belong to `species`."""
        return 100 * float((self.penguins["species"] == species).mean())


with Logger():
    reply = invoke(
        prompt="What share of the penguins on Dream are Chinstrap, in percent?",
        dream=Colony("Dream", penguins[penguins["island"] == "Dream"]),
    )

reply

--- USER PROMPT BEGINS ---
<prompt type="str">
What share of the penguins on Dream are Chinstrap, in percent?
</prompt>
<dream type="Colony">
`dream` [Colony]: The penguins observed on one island.
  - `island` [str]
  - `penguins` [DataFrame]
  - `share(species: str) -> float`: Percentage of this colony's penguins that belong to `species`.
</dream>
The input variables `prompt`, `dream` are available in your REPL with descriptions given above.
They are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.

--- USER PROMPT ENDS ---


--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute Dream colony share of Chinstrap penguins and return it in percent.
chinstrap_pct = dream.share("Chinstrap")
return chinstrap_pct
--- LLM RESPONSE ENDS ---


54.83870967741935

`Colony` was never registered or described anywhere. The model was shown its docstring,
its two attributes with their types, and its one method with its signature — and that was
enough to work out that it should call `dream.share(...)`.

## What to take away

- **One mechanism.** A prompt, a tool, a DataFrame, your own object — all keyword
  arguments, all variables in the REPL, all shown in the prompt.
  - The input's name is its variable name in the REPL.
  - The input's name is the title of the section displaying the input.
- **A function's display is its signature and docstring.** Giving the agent a tool amounts
  to passing a function with a docstring.
- **An object's display is its class docstring and its public members** — unless it already
  prints itself, in which case the model is shown the value.

Writing a clear docstring is therefore the whole of "prompt engineering" for a tool or an
object: the text you write for human readers is the text the model reads too.

---

**Next:** [Recursive subagents and scope](02-subagents-and-scope.ipynb) — an agent can call `invoke` itself, which raises the question of what a *subagent* can see.